# Hybrid Ant Colony Optimization (ACO) for the Traveling Salesman Problem (TSP)
**Author:** @VedantAndhale


## 1. Define Values

In [ ]:
import numpy as np
import random

# Define the distance matrix (distances between cities)
distance_matrix = np.array(
    [[0, 10, 18, 20], [10, 0, 35, 25], [15, 35, 0, 30], [20, 25, 30, 0]]
)

# Parameters for Ant Colony Optimization
num_ants = 12
num_iterations = 50
evaporation_rate = 0.5
pheromone_constant = 1.0
heuristic_constant = 1.0

# Initialize pheromone matrix and visibility matrix
num_cities = len(distance_matrix)
pheromone = np.ones((num_cities, num_cities))  # Pheromone matrix
visibility = np.zeros_like(distance_matrix, dtype=float)  # Visibility matrix

# Compute visibility (inverse of distance, except for zero distances)
non_zero_distances = distance_matrix > 0
visibility[non_zero_distances] = 1 / distance_matrix[non_zero_distances]

## 2. 2-opt Local Search Function
The 2-opt algorithm is used to improve a given route by iteratively swapping two edges to reduce the total distance.

In [ ]:
def two_opt(route, distance_matrix):
    """Apply 2-opt local search to improve the given route."""
    best = route
    improved = True
    while improved:
        improved = False
        for i in range(1, len(route) - 2):
            for j in range(i + 1, len(route)):
                if j - i == 1:
                    continue
                new_route = best[:]
                new_route[i:j] = best[j - 1 : i - 1 : -1]
                if route_distance(new_route, distance_matrix) < route_distance(
                    best, distance_matrix
                ):
                    best = new_route
                    improved = True
        route = best
    return best


def route_distance(route, distance_matrix):
    return sum(
        distance_matrix[route[i]][route[(i + 1) % len(route)]]
        for i in range(len(route))
    )

## 3. Hybrid ACO Algorithm with 2-opt Local Search
Each ant constructs a tour based on pheromone and visibility, then the 2-opt algorithm is applied to improve the tour. Pheromone trails are updated after each iteration.

In [27]:
# Hybrid ACO algorithm with 2-opt local search
for iteration in range(num_iterations):
    ant_routes = []
    for ant in range(num_ants):
        current_city = random.randint(0, num_cities - 1)
        visited_cities = [current_city]
        route = [current_city]

        while len(visited_cities) < num_cities:
            probabilities = []
            for city in range(num_cities):
                if city not in visited_cities:
                    pheromone_value = pheromone[current_city][city]
                    visibility_value = visibility[current_city][city]
                    probability = (pheromone_value**pheromone_constant) * (
                        visibility_value**heuristic_constant
                    )
                    probabilities.append((city, probability))

            # Normalize probabilities
            total = sum(prob for _, prob in probabilities)
            if total == 0:
                selected_city = random.choice([city for city, _ in probabilities])
            else:
                probs = [prob / total for _, prob in probabilities]
                selected_city = random.choices(
                    [city for city, _ in probabilities], weights=probs
                )[0]

            route.append(selected_city)
            visited_cities.append(selected_city)
            current_city = selected_city

        # Apply 2-opt local search to improve the route
        improved_route = two_opt(route, distance_matrix)
        ant_routes.append(improved_route)

    # Update pheromone levels
    delta_pheromone = np.zeros((num_cities, num_cities))
    for route in ant_routes:
        for i in range(len(route) - 1):
            city_a = route[i]
            city_b = route[i + 1]
            delta_pheromone[city_a][city_b] += 1 / distance_matrix[city_a][city_b]
            delta_pheromone[city_b][city_a] += 1 / distance_matrix[city_a][city_b]

    pheromone = (1 - evaporation_rate) * pheromone + delta_pheromone

## 4. Extracting the Best Route and Its Length
After all iterations, select the best route found by the ants and calculate its total distance.

In [ ]:
# Find the best route
best_route_index = np.argmin(
    [route_distance(cities, distance_matrix) for cities in ant_routes]
)
best_route = ant_routes[best_route_index]
shortest_distance = route_distance(best_route, distance_matrix)

print("Best route:", best_route)
print("Shortest distance:", shortest_distance)